# Znajdowanie czynników popytu za pomocą PROC GLMSELECT: selekcja krokowa, LASSO i zwalidowana selekcja postępująca

## Podsumowanie wykonawcze

Zespół analityki detalicznej używa **PROC GLMSELECT**, aby odkryć, które dźwignie marketingowe i cenowe faktycznie napędzają tygodniową sprzedaż jednostek, oddzielając prawdziwe czynniki popytu od szumu. Selekcja krokowa oceniana przez SBC, LASSO z 5-krotną walidacją krzyżową oraz selekcja postępująca zwalidowana na zbiorze wstrzymanym — wszystkie odzyskują **ten sam zestaw prawdziwych czynników** — cenę własną, cenę konkurencji, wydatki reklamowe, liczbę e-maili, promocje, święta, region `Northeast` oraz kanał `Online` — a każda z czterech zasadzonych zmiennych-wabików (`temp_f`, `noise1`, `noise2`, `noise3`) jest automatycznie odrzucana.

Metody zgadzają się też ściśle co do wielkości efektów: każda szacuje efekt ceny własnej bliski **-28 jednostek na dolara** oraz efekt ceny konkurencji bliski **+14** — znaki substytucji wbudowane w równanie generujące dane. Jedyna niezgoda jest na marginesie — LASSO z walidacją krzyżową dodatkowo zachowuje mały, statystycznie nieistotny kontrast `Region=Midwest` (oszacowanie -8.3 z błędem standardowym 8.3), który zarówno selekcja krokowa, jak i postępująca odrzucają. Lista czynników, która przetrwa trzy różne filozofie selekcji, jest znacznie bardziej wiarygodna niż taka dopasowana do jednej reguły.

## Źródła danych

Wszystkie dane w tym notatniku są **syntetyczne** i generowane w locie (bez plików zewnętrznych, ziarno `20260531`). Odwzorowują jeden sezon paneli popytu sklep-tydzień dla detalisty dóbr konsumpcyjnych.

| Zbiór danych | Wiersze | Ziarnistość | Kluczowe zmienne |
|---------|------|-------|---------------|
| `demand` | 100 | Sklep-tydzień | `units` (odpowiedź: tygodniowa sprzedaż jednostek); `price`, `competitor` (cena własna i cena rywala na półce); `adspend`, `email` (presja medialna); `promo`, `holiday` (flagi zdarzeń); `region`, `channel` (efekty CLASS); `temp_f`, `noise1`-`noise3` (predyktory-wabiki/nieistotne) |

`units` jest zbudowana ze znanego równania liniowego, dzięki czemu "poprawny" zestaw czynników jest weryfikowalny; `temp_f` oraz trzy kolumny `noise` nie niosą prawdziwego sygnału i istnieją, aby sprawdzić, czy każda metoda selekcji je odrzuci.

# Znajdowanie czynników popytu za pomocą PROC GLMSELECT

Kierownik kategorii ma dziesiątki kandydujących zmiennych objaśniających dla tygodniowej sprzedaży: cenę własną, cenę konkurencji, wydatki reklamowe, liczbę e-maili, promocje, święta, region sklepu, kanał sprzedaży, a nawet pogodę. Wrzucenie ich wszystkich do jednej regresji zaprasza przeuczenie i niestabilne współczynniki. **PROC GLMSELECT** automatyzuje poszukiwanie oszczędnego modelu, wspierając selekcję krokową, LASSO, elastic net oraz selekcję najmniejszego kąta, z wbudowaną walidacją krzyżową i podziałem na zbiór wstrzymany.

W tym notatniku:

1. Generujemy realistyczny syntetyczny panel popytu sklep-tydzień ze *znanym* zestawem prawdziwych czynników (plus celowe zmienne-wabiki).
2. Uruchamiamy **selekcję krokową** ocenianą przez SBC.
3. Uruchamiamy **LASSO** z 5-krotną walidacją krzyżową.
4. Uruchamiamy **selekcję postępującą zwalidowaną na 30% zbiorze wstrzymanym**.

Dobra metoda selekcji powinna odzyskać prawdziwe czynniki i odrzucić szum — zobaczmy.

## 1. Wygenerowanie syntetycznego panelu popytu

Odpowiedź `units` jest skonstruowana z jawnego równania liniowego, więc znamy prawdę podstawową: cena i cena konkurencji, wydatki reklamowe, liczba e-maili, flagi promocji i święta, a także efekty regionu i kanału — wszystkie mają znaczenie. Zmienne `temp_f`, `noise1`, `noise2` i `noise3` to czyste wabiki bez prawdziwego związku ze sprzedażą. Ziarno `call streaminit` sprawia, że dane są odtwarzalne.

In [1]:
/* ---------------------------------------------------------------
   Syntetyczny panel popytu sklep-tydzień dla detalisty dóbr konsumpcyjnych.
   'units' wynika ze ZNANEGO równania; temp_f i noise1-3 to zmienne-wabiki.
   --------------------------------------------------------------- */
DANE demand;
    CALL streaminit(20260531);
    DŁUGOŚĆ region $9 channel $8 promo $3;
    POWTÓRZ store_week = 1 TO 100;
        /* Mieszanka regionów */
        u1 = rand('uniform');
        JEŚLI u1 < 0.34 WTEDY region = 'Northeast';
        PRZECIWNIE JEŚLI u1 < 0.67 WTEDY region = 'Midwest';
        PRZECIWNIE region = 'South';

        /* Kanał sprzedaży */
        JEŚLI rand('uniform') < 0.45 WTEDY channel = 'Store';
        PRZECIWNIE channel = 'Online';

        /* Czynniki cenowe i medialne */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Flagi zdarzeń i nieistotny wabik pogodowy */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        JEŚLI rand('uniform') < 0.40 WTEDY promo = 'Yes';
        PRZECIWNIE promo = 'No';

        /* Czyste predyktory szumu (bez prawdziwego efektu) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Tygodniowa sprzedaż jednostek ze znanego równania strukturalnego */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        JEŚLI units < 0 WTEDY units = 0;
        WYJŚCIE;
    KONIEC;
    ZACHOWAJ region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
WYKONAJ;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profilowanie danych

Przed modelowaniem potwierdź, że odpowiedź i główne ciągłe kandydatki są na sensownych skalach.

In [2]:
PROCEDURA ŚREDNIE DANE=demand n mean std MIN MAX maxdec=1;
    ZMIENNA units price competitor adspend email;
    ETYKIETA units="Sprzedane jednostki (tygodniowo)" price="Cena własna"
          competitor="Cena konkurencji" adspend="Wydatki reklamowe"
          email="Liczba e-maili";
    TYTUŁ "Tygodniowy popyt i kandydujące czynniki";
WYKONAJ;

                                        Tygodniowy popyt i kandydujące czynniki                                         

                                                  The MEANS Procedure

 Variable    Label                                    N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------------------
 units       Sprzedane jednostki (tygodniowo)       100       874.8       136.3       508.6      1162.3
 price       Cena własna                            100        14.0         3.4         8.0        19.9
 competitor  Cena konkurencji                       100        13.8         3.4         8.1        19.9
 adspend     Wydatki reklamowe                      100       990.6       726.9        50.0      3358.0
 email       Liczba e-maili                         100        45.5        26.4         0.0        99.0
 ------------------------------------------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Selekcja krokowa oceniana przez SBC

Selekcja krokowa naprzemiennie dodaje i usuwa efekty, tutaj oceniana przez **kryterium bayesowskie Schwarza (SBC)** zarówno dla testu wejścia (`select=sbc`), jak i ostatecznego wyboru modelu (`choose=sbc`). SBC karze złożoność mocniej niż AIC, faworyzując szczuplejsze modele.

Kluczowe instrukcje i opcje:

- `CLASS region channel promo / param=reference` deklaruje predyktory kategoryczne z kodowaniem komórki referencyjnej.
- `selection=stepwise(select=sbc choose=sbc)` napędza poszukiwanie.
- `details=summary` drukuje krok po kroku podsumowanie selekcji; `stb` dodaje standaryzowane współczynniki, aby efekty na różnych skalach były porównywalne.
- `output out=demand_scored p=predicted r=residual` zapisuje wartości dopasowane i reszty do dalszego punktowania.

Odczytaj **Podsumowanie selekcji krokowej** jako ślad poszukiwania: zaczyna od pełnego modelu z 12 efektami i *usuwa* efekty jeden po drugim, odrzucając kolejno `noise1`, `noise2`, `temp_f`, kontrast `Region=Midwest` i `noise3`, ponieważ każde usunięcie obniża SBC. To, co przetrwa w tabeli **Oszacowania parametrów**, jest wybranym modelem.

In [3]:
PROCEDURA glmselect DANE=demand seed=20260531;
    KLASA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(WYBIERZ=sbc choose=sbc)
          details=summary stb;
    ETYKIETA units="Sprzedane jednostki (tygodniowo)" region="Region" channel="Kanał"
          promo="Promocja" price="Cena własna" competitor="Cena konkurencji"
          adspend="Wydatki reklamowe" email="Liczba e-maili"
          temp_f="Temperatura (°F)" holiday="Święto"
          noise1="Szum 1" noise2="Szum 2" noise3="Szum 3";
    WYJŚCIE out=demand_scored p=predicted r=residual;
    TYTUŁ "Selekcja krokowa czynników popytu (SBC)";
WYKONAJ;

                                        Tygodniowy popyt i kandydujące czynniki                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Sprzedane jednostki (tygodniowo)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                  Stepwise Selection Summary                                                  

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  -----------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed             Szum 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  7


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO z walidacją krzyżową

LASSO ściąga współczynniki w kierunku zera, wykonując selekcję i regularyzację jednocześnie. Zamiast zatrzymywać się na stałym kryterium, pozwalamy, aby **5-krotna walidacja krzyżowa** wybrała punkt na ścieżce LASSO z najlepszym błędem poza próbą.

- `selection=lasso(choose=cv stop=none)` śledzi całą ścieżkę LASSO i wybiera krok optymalny wg walidacji krzyżowej.
- `cvmethod=random(5)` przypisuje obserwacje do 5 losowych podzbiorów.

**Podsumowanie selekcji LASSO** pokazuje kolejność, w jakiej efekty wchodzą w miarę rozluźniania kary: najpierw `price`, potem `adspend`, `competitor`, region `Northeast`, `email`, `promo` i `holiday` — wszystkie siedem prawdziwych sygnałów wchodzi przed jakimkolwiek wabikiem. Walidacja krzyżowa następnie wybiera krok, w którym błąd PRESS poza próbą jest najniższy, a wynikowa tabela **Oszacowania parametrów** zachowuje dokładnie prawdziwe czynniki (plus mały człon `Region=Midwest`), wykluczając `temp_f`, `noise1`, `noise2` i `noise3`, które wchodzą dopiero na samym końcu ścieżki.

In [4]:
PROCEDURA glmselect DANE=demand seed=20260531;
    KLASA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv ZATRZYMAJ=none)
          cvmethod=RANDOM(5);
    ETYKIETA units="Sprzedane jednostki (tygodniowo)" region="Region" channel="Kanał"
          promo="Promocja" price="Cena własna" competitor="Cena konkurencji"
          adspend="Wydatki reklamowe" email="Liczba e-maili"
          temp_f="Temperatura (°F)" holiday="Święto"
          noise1="Szum 1" noise2="Szum 2" noise3="Szum 3";
    TYTUŁ "LASSO z 5-krotną walidacją krzyżową";
WYKONAJ;

                                        Tygodniowy popyt i kandydujące czynniki                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Sprzedane jednostki (tygodniowo)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                          LASSO Selection Summary                                                          

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------- 


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Selekcja postępująca zwalidowana na zbiorze wstrzymanym

Trzecia, uzupełniająca strategia: zbuduj model przez **selekcję postępującą** (efekty tylko wchodzą, nigdy nie wychodzą), ale zatrzymaj się tam, gdzie wydajność na niezależnej próbie wstrzymanej jest najlepsza — bezpośrednio chroniąc przed przeuczeniem.

- `partition fraction(validate=0.30)` losowo rezerwuje 30% wierszy do walidacji, pozostawiając 70 obserwacji treningowych i 30 walidacyjnych.
- `selection=forward(select=aic choose=validate)` dodaje efekty wg AIC, ale wybiera model końcowy wg błędu na próbie walidacyjnej.

Tabela **Frakcje podziału** potwierdza podział 70/30. Selekcja postępująca następnie wprowadza efekty, dopóki błąd walidacyjny przestaje się poprawiać; osiem efektów w końcowej tabeli **Oszacowania parametrów** to dokładnie prawdziwe czynniki, przy czym cztery wabiki nigdy nie zostają dopuszczone. Gdy trzy metody zbudowane na różnych zasadach trafiają na te same czynniki, model jest znacznie bardziej prawdopodobnie prawdziwy niż artefakt jednej reguły selekcji.

In [5]:
PROCEDURA glmselect DANE=demand seed=20260531;
    KLASA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(WYBIERZ=aic choose=validate);
    partition FRACTION(validate=0.30);
    ETYKIETA units="Sprzedane jednostki (tygodniowo)" region="Region" channel="Kanał"
          promo="Promocja" price="Cena własna" competitor="Cena konkurencji"
          adspend="Wydatki reklamowe" email="Liczba e-maili"
          temp_f="Temperatura (°F)" holiday="Święto"
          noise1="Szum 1" noise2="Szum 2" noise3="Szum 3";
    TYTUŁ "Selekcja postępująca zwalidowana na 30% zbiorze wstrzymanym";
WYKONAJ;

                                        Tygodniowy popyt i kandydujące czynniki                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Sprzedane jednostki (tygodniowo)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         Forward Selection Summary                                                         

    Step    Action             Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  -


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpretacja wyników

Wszystkie trzy strategie selekcji odzyskują **ten sam zestaw prawdziwych czynników popytu** i odrzucają każdy wabik. Czytając wprost z tabel **Oszacowania parametrów**:

- **Cena** jest dominującym czynnikiem. Jej standaryzowany współczynnik w modelu krokowym wynosi **-0.70**, zdecydowanie największy co do wartości bezwzględnej, a surowe nachylenie mieści się między **-27.8** (krokowa i LASSO) a **-28.6** (postępująca) jednostek na dolara — niemal dokładnie -28, z jakim wygenerowano dane. **Cena konkurencji** przesuwa popyt w przeciwnym kierunku o około **+14.4** we wszystkich trzech dopasowaniach — zachowanie substytucyjne, jakiego oczekuje kierownik kategorii.
- **Wydatki reklamowe** (około +0.062 jednostki na dolara) i **liczba e-maili** (około +1.2 jednostki na wysyłkę) obie podnoszą sprzedaż, kwantyfikując reakcję na media.
- **Promocje** i **święta** niosą największe efekty zdarzeń: kontrast `Promo=No` wynosi około **-51 do -57** względem tygodnia promowanego, a wzrost świąteczny to około **+66 do +76** jednostek. Region `Northeast` (około +49 do +55) i kanał `Online` (około +24 do +32) niosą dodatnie efekty bazowe.
- Co kluczowe, każdy wabik — `temp_f`, `noise1`, `noise2`, `noise3` — jest **odrzucany** przez selekcję krokową i postępującą oraz wykluczony z modelu LASSO wybranego przez walidację krzyżową. Każda metoda odzyskała sygnał strukturalny i zignorowała szum.

Trzy metody nie są identyczne bit po bicie: selekcja krokowa (SBC) i postępujące poszukiwanie zwalidowane na zbiorze wstrzymanym osiadają na tych samych ośmiu efektach, podczas gdy LASSO z walidacją krzyżową dodatkowo zatrzymuje mały kontrast `Region=Midwest` (-8.3, błąd standardowy 8.3) — różnica na poziomie szumu, a nie istotna niezgoda. Fakt, że lista czynników przetrwa krokową SBC, LASSO z walidacją krzyżową i postępującą selekcję zwalidowaną na zbiorze wstrzymanym, jest prawdziwym wnioskiem: odkrycie odporne na trzy różne filozofie selekcji jest znacznie bardziej wiarygodne niż jedno dopasowane do pojedynczego kryterium. Dzięki `OUTPUT OUT=demand_scored` przechwytującemu wartości dopasowane i reszty, ten sam przepływ pracy naturalnie rozszerza się na punktowanie planowanego na przyszły kwartał kalendarza cen i promocji.